In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()

while not (PROJECT_ROOT / "src").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("找不到项目根目录（包含 src 的目录）")
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT =", PROJECT_ROOT)

PROJECT_ROOT = /home/joyce/projects/cross-asset-quant-lab/cryptoAlpha


In [3]:
import pandas as pd
import numpy as np

from src.factors.factor_builder import FactorBuilder
from src.models.xgb_alpha_model import XGBAlphaModel
from src.portfolio.portfolio_backtest import (
    backtest_long_short_portfolio,
    summarize_portfolio_result,
)
from src.portfolio.portfolio_plotter import PortfolioPlotter

In [7]:
factor_names = [
    "mom_24h",
    "mom_6h",
    "funding_z_24",
    "oi_change_24h",
    "taker_imbalance",
    "long_short_ratio_z_24",
    "volume_ratio_24",
    "active_community_count_z_24",
]
import pandas as pd
cache_dir = PROJECT_ROOT / "data" / "cache"
cache_dir.mkdir(parents=True, exist_ok=True)

panel_fp = cache_dir / "panel_top20_1h_2025_20260311.parquet"
panel_df = pd.read_parquet(panel_fp)
builder = FactorBuilder()
factor_df = builder.compute_many(
    panel_df,
    factor_names
)

print(factor_df.head())
print(factor_df.shape)

/home/joyce/projects/cross-asset-quant-lab/cryptoAlpha/src/factors/factor_utils.py:32: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  return df.groupby("symbol")[col].pct_change(periods)


    datetime    symbol  mom_24h  mom_6h  funding_z_24  oi_change_24h  \
0 2025-01-01   ADAUSDT      NaN     NaN           NaN            NaN   
1 2025-01-01   APTUSDT      NaN     NaN           NaN            NaN   
2 2025-01-01  ATOMUSDT      NaN     NaN           NaN            NaN   
3 2025-01-01  AVAXUSDT      NaN     NaN           NaN            NaN   
4 2025-01-01   BCHUSDT      NaN     NaN           NaN            NaN   

   taker_imbalance  long_short_ratio_z_24  volume_ratio_24  \
0         0.224374                    NaN              NaN   
1         0.117690                    NaN              NaN   
2         0.014625                    NaN              NaN   
3         0.151342                    NaN              NaN   
4         0.078970                    NaN              NaN   

   active_community_count_z_24  
0                          NaN  
1                          NaN  
2                          NaN  
3                          NaN  
4                          Na

In [8]:

model = XGBAlphaModel(
    horizon=24,       # 预测未来24h收益
    train_window=180, # 训练窗口，按时间点滚动
)

# 先在 panel 上构造 label
df_model = model.build_label(panel_df)

# 再 merge 因子
df_model = df_model.merge(
    factor_df,
    on=["datetime", "symbol"],
    how="left",
)

print(df_model.shape)
df_model.head()

(197923, 36)


,datetime,symbol,open,high,low,close,volume_usd,funding_open,funding_high,funding_low,...,negative_sentiment_ratio,future_return,mom_24h,mom_6h,funding_z_24,oi_change_24h,taker_imbalance,long_short_ratio_z_24,volume_ratio_24,active_community_count_z_24
0,2025-01-01,ADAUSDT,0.8448,0.8607,0.8434,0.8593,1.443051e+07,0.01,0.01,0.01,...,NaN,0.100314,NaN,NaN,NaN,NaN,0.224374,NaN,NaN,NaN
1,2025-01-01,APTUSDT,8.7120,8.8049,8.7014,8.8004,5.540888e+06,0.01,0.01,0.01,...,NaN,0.036998,NaN,NaN,NaN,NaN,0.117690,NaN,NaN,NaN
2,2025-01-01,ATOMUSDT,6.1870,6.2810,6.1770,6.2790,1.511668e+06,0.01,0.01,0.01,...,NaN,0.063227,NaN,NaN,NaN,NaN,0.014625,NaN,NaN,NaN
3,2025-01-01,AVAXUSDT,35.6890,36.2210,35.6310,36.1900,8.185929e+06,0.01,0.01,0.01,...,NaN,0.063830,NaN,NaN,NaN,NaN,0.151342,NaN,NaN,NaN
4,2025-01-01,BCHUSDT,434.3500,440.7100,433.7300,440.5900,3.551392e+06,0.01,0.01,0.01,...,NaN,0.036746,NaN,NaN,NaN,NaN,0.078970,NaN,NaN,NaN


In [ ]:
pred_df = model.fit_predict(
    df=df_model,
    feature_cols=factor_names,
    label_col="future_return",
)

print(pred_df.shape)
pred_df.head()

In [ ]:
pred_dir = PROJECT_ROOT / "data" / "predictions"
pred_dir.mkdir(parents=True, exist_ok=True)

pred_fp = pred_dir / "pred_df_top20_1h_2025_20260311.parquet"
pred_df.to_parquet(pred_fp, index=False)

print("saved pred_df to:", pred_fp)
pred_df.head()